In [1]:
import scanpy as sc
import numpy as np 
import pandas as pd 
import anndata as ad
from glob import glob
from tqdm import tqdm
from multiprocessing import Pool
import seaborn as sns
sc._settings.ScanpyConfig.n_jobs=48

# 0. Check the QC results and remove low quality sample.

In [2]:
# D0689 and D0503

# 800Gb memory size is required

# 1. Concat filtered scADT data

In [3]:
files = glob('/home/liyanguo/MyImmuCell/02_Read_QC/MyImmuCell_ADT_h5ad/*_ADT_filtered.h5ad')

In [4]:
files[1:10]

['/home/liyanguo/MyImmuCell/02_Read_QC/MyImmuCell_ADT_h5ad/D0079_Rep2_ADT_filtered.h5ad',
 '/home/liyanguo/MyImmuCell/02_Read_QC/MyImmuCell_ADT_h5ad/D0246_Rep1_ADT_filtered.h5ad',
 '/home/liyanguo/MyImmuCell/02_Read_QC/MyImmuCell_ADT_h5ad/D0100_Rep2_ADT_filtered.h5ad',
 '/home/liyanguo/MyImmuCell/02_Read_QC/MyImmuCell_ADT_h5ad/D0170_Rep2_ADT_filtered.h5ad',
 '/home/liyanguo/MyImmuCell/02_Read_QC/MyImmuCell_ADT_h5ad/D0235_Rep1_ADT_filtered.h5ad',
 '/home/liyanguo/MyImmuCell/02_Read_QC/MyImmuCell_ADT_h5ad/D0083_Rep1_ADT_filtered.h5ad',
 '/home/liyanguo/MyImmuCell/02_Read_QC/MyImmuCell_ADT_h5ad/D0925_Rep2_ADT_filtered.h5ad',
 '/home/liyanguo/MyImmuCell/02_Read_QC/MyImmuCell_ADT_h5ad/D0471_Rep1_ADT_filtered.h5ad',
 '/home/liyanguo/MyImmuCell/02_Read_QC/MyImmuCell_ADT_h5ad/D0073_Rep1_ADT_filtered.h5ad']

In [5]:
len(files)

1994

In [6]:
data_dic = {}
for i in tqdm(files):
    sample_name = i.split('/')[-1].split('.')[0]
    adata = sc.read_h5ad(i)
    data_dic[sample_name] = adata

100%|███████████████████████████████████████████████████████████████████████████████████████| 1994/1994 [01:22<00:00, 24.25it/s]


In [7]:
adata = sc.concat(data_dic)

In [8]:
adata

AnnData object with n_obs × n_vars = 69161759 × 30
    obs: 'orig.ident', 'SampleID', 'DonorID'

In [9]:
adata.obs

,orig.ident,SampleID,DonorID
D0325_Rep1_CACCTTAC_AGTCACTA_AGCCATGC,CACCTTAC,D0325_Rep1,D0325
D0325_Rep1_TATCAGCA_TCTTCACA_ACCTCCAA,TATCAGCA,D0325_Rep1,D0325
D0325_Rep1_AAACATCG_CATACCAA_GTCTGTCA,AAACATCG,D0325_Rep1,D0325
D0325_Rep1_CCTAATCC_GAACAGGC_AAACATCG,CCTAATCC,D0325_Rep1,D0325
D0325_Rep1_ACGTATCA_AGGCTAAC_GAGCTGAA,ACGTATCA,D0325_Rep1,D0325
...,...,...,...
D0443_Rep2_CCTCTATC_AGAGTCAA_GCGAGTAA,CCTCTATC,D0443_Rep2,D0443
D0443_Rep2_ACATTGGC_AAGAGATC_AGGCTAAC,ACATTGGC,D0443_Rep2,D0443
D0443_Rep2_CACCTTAC_GTGTTCTA_CAAGGAGC,CACCTTAC,D0443_Rep2,D0443
D0443_Rep2_GACAGTGC_AACCGAGA_TGAAGAGA,GACAGTGC,D0443_Rep2,D0443


In [10]:
adata.write_h5ad("/home/liyanguo/MyImmuCell/02_Read_QC/scADT_filtered_MyImmuCell.h5ad",compression="gzip")

In [11]:
print ('scADT filtered Done.')

scADT filtered Done.


In [12]:
del adata,data_dic

# 2. Concat scRNA data of low UMI detection

In [13]:
files = glob('/home/liyanguo/MyImmuCell/02_Read_QC/MyImmuCell_RNA_h5ad/*_low_nCount_RNA.h5ad')

In [14]:
files[1:10]

['/home/liyanguo/MyImmuCell/02_Read_QC/MyImmuCell_RNA_h5ad/D0110_E_Rep1_low_nCount_RNA.h5ad',
 '/home/liyanguo/MyImmuCell/02_Read_QC/MyImmuCell_RNA_h5ad/D0369_Rep2_low_nCount_RNA.h5ad',
 '/home/liyanguo/MyImmuCell/02_Read_QC/MyImmuCell_RNA_h5ad/D0843_Rep1_low_nCount_RNA.h5ad',
 '/home/liyanguo/MyImmuCell/02_Read_QC/MyImmuCell_RNA_h5ad/D0550_Rep2_low_nCount_RNA.h5ad',
 '/home/liyanguo/MyImmuCell/02_Read_QC/MyImmuCell_RNA_h5ad/D0421_Rep1_low_nCount_RNA.h5ad',
 '/home/liyanguo/MyImmuCell/02_Read_QC/MyImmuCell_RNA_h5ad/D0210_Rep1_low_nCount_RNA.h5ad',
 '/home/liyanguo/MyImmuCell/02_Read_QC/MyImmuCell_RNA_h5ad/D0181_Rep2_low_nCount_RNA.h5ad',
 '/home/liyanguo/MyImmuCell/02_Read_QC/MyImmuCell_RNA_h5ad/D0354_Rep2_low_nCount_RNA.h5ad',
 '/home/liyanguo/MyImmuCell/02_Read_QC/MyImmuCell_RNA_h5ad/D0906_E_Rep1_low_nCount_RNA.h5ad']

In [15]:
len(files)

1994

In [16]:
chunk_size = 100
merged_adata = None

In [17]:
for chunk_start in range(0,len(files),chunk_size):
    data_dic = {}
    chunk_files = files[chunk_start:chunk_start+chunk_size]
    
    #read chunk
    for i in tqdm(chunk_files):
        sample_name = i.split('/')[-1].split('.')[0]
        adata = sc.read_h5ad(i)
        data_dic[sample_name] = adata
    chunk_adata = sc.concat(data_dic,join='outer')
    del data_dic
    
    if merged_adata is None:
            merged_adata = chunk_adata
    else:
        merged_adata = sc.concat([merged_adata,chunk_adata],join='outer')
    del chunk_adata
print ('Read chunk done.')

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 94/94 [00:54<00:00,  1.74it/s]


Read chunk done.


In [18]:
merged_adata

AnnData object with n_obs × n_vars = 36453806 × 38606
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'SampleID', 'DonorID', 'percent_mito', 'percent_ribo', 'percent_mito_ribo', 'log10GenesPerUMI', 'percent_top50', 'percent_oxphos', 'percent_apop', 'percent_dna_repair', 'percent_ieg', 'percent_hemo', 'S.Score', 'G2M.Score', 'Phase', 'Reference_Atlas_L1L2_mv', 'Reference_Atlas_L1L2_pl', 'AIFI_L1', 'AIFI_L2', 'AIFI_L3', 'Immune_All_High', 'Immune_All_Low', 'Adult_Human_Blood', 'Adult_Human_Bone_marrow', 'percent.mt', 'predicted.celltype.l1.score', 'predicted.celltype.l1', 'predicted.celltype.l2.score', 'predicted.celltype.l2', 'predicted.celltype.l3.score', 'predicted.celltype.l3', 'mapping.score', 'scDblFinder.class'

In [19]:
merged_adata.obs

,orig.ident,nCount_RNA,nFeature_RNA,SampleID,DonorID,percent_mito,percent_ribo,percent_mito_ribo,log10GenesPerUMI,percent_top50,...,Adult_Human_Bone_marrow,percent.mt,predicted.celltype.l1.score,predicted.celltype.l1,predicted.celltype.l2.score,predicted.celltype.l2,predicted.celltype.l3.score,predicted.celltype.l3,mapping.score,scDblFinder.class
D0213_Rep1_CCGAAGTA_ACAGCAGA_ATCCTGTA,CCGAAGTA,17076.0,4234,D0213_Rep1,D0213,3.859218,17.732490,21.591708,0.856905,23.325135,...,Classical monocytes,3.859218,1.000000,Mono,1.000000,CD14 Mono,1.000000,CD14 Mono,1.000000,singlet
D0213_Rep1_CAAGACTA_AACGCTTA_CAACCACA,CAAGACTA,16300.0,4070,D0213_Rep1,D0213,4.613497,9.288344,13.901840,0.856941,25.159509,...,Classical monocytes,4.613497,1.000000,Mono,1.000000,CD14 Mono,1.000000,CD14 Mono,0.951222,singlet
D0213_Rep1_AAACATCG_CTGGCATA_TGGCTTCA,AAACATCG,15957.0,3649,D0213_Rep1,D0213,5.301748,15.184559,20.486307,0.847541,28.482798,...,Classical monocytes,5.301748,1.000000,Mono,1.000000,CD14 Mono,1.000000,CD14 Mono,0.960374,singlet
D0213_Rep1_ACCTCCAA_TGGTGGTA_CGAACTTA,ACCTCCAA,15754.0,3835,D0213_Rep1,D0213,5.535102,13.418814,18.953916,0.853808,27.897677,...,Classical monocytes,5.535102,1.000000,Mono,0.599979,CD14 Mono,0.599979,CD14 Mono,1.000000,singlet
D0213_Rep1_TCTTCACA_CCTCCTGA_AACGTGAT,TCTTCACA,15722.0,4257,D0213_Rep1,D0213,3.288386,10.361277,13.649663,0.864791,20.868846,...,Classical monocytes,3.288386,1.000000,Mono,1.000000,CD14 Mono,1.000000,CD14 Mono,1.000000,singlet
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
D0955_M_Rep2_GAATCTGA_AGCCATGC_CCAGTTCA,GAATCTGA,340.0,252,D0955_M_Rep2,D0955,0.294118,4.411765,4.705882,0.948616,40.588235,...,NAMPT neutrophils,0.294118,0.326632,CD4 T,0.283673,CD14 Mono,0.283673,CD14 Mono,0.138567,singlet
D0955_M_Rep2_CCATCCTC_ACAGCAGA_CAGCGTTA,CCATCCTC,335.0,250,D0955_M_Rep2,D0955,0.895522,2.985075,3.880597,0.949662,40.298507,...,NAMPT neutrophils,0.895522,0.341799,other,0.239507,Platelet,0.239507,Platelet,0.169108,singlet
D0955_M_Rep2_GTACGCAA_CCGACAAC_GGTGCGAA,GTACGCAA,303.0,200,D0955_M_Rep2,D0955,0.000000,5.940594,5.940594,0.927295,49.504950,...,NAMPT neutrophils,0.000000,0.309592,other,0.275081,CD14 Mono,0.275081,CD14 Mono,0.121042,singlet
D0955_M_Rep2_AGAGTCAA_AGCCATGC_TGGAACAA,AGAGTCAA,287.0,219,D0955_M_Rep2,D0955,0.000000,1.742160,1.742160,0.952220,41.114983,...,NAMPT neutrophils,0.000000,0.321042,CD4 T,0.288080,CD4 TCM,0.276521,CD4 TCM_1,0.111093,singlet


In [20]:
merged_adata.write_h5ad("/home/liyanguo/MyImmuCell/02_Read_QC/scRNA_MyImmuCell_low_nCount_RNA.h5ad",compression="gzip")

In [21]:
print ('scRNA Done.')

scRNA Done.


In [22]:
del merged_adata

# 3. Concat scRNA data of high UMI detection

In [ ]:
files = glob('/home/liyanguo/MyImmuCell/02_Read_QC/MyImmuCell_RNA_h5ad/*_high_nCount_RNA.h5ad')

In [ ]:
files[1:10]

In [ ]:
len(files)

In [ ]:
chunk_size = 100
merged_adata = None

In [ ]:
for chunk_start in range(0,len(files),chunk_size):
    data_dic = {}
    chunk_files = files[chunk_start:chunk_start+chunk_size]
    
    #read chunk
    for i in tqdm(chunk_files):
        sample_name = i.split('/')[-1].split('.')[0]
        adata = sc.read_h5ad(i)
        data_dic[sample_name] = adata
    chunk_adata = sc.concat(data_dic,join='outer')
    del data_dic
    
    if merged_adata is None:
            merged_adata = chunk_adata
    else:
        merged_adata = sc.concat([merged_adata,chunk_adata],join='outer')
    del chunk_adata
print ('Read chunk done.')

In [ ]:
merged_adata

In [ ]:
merged_adata.obs

In [ ]:
merged_adata.write_h5ad("/home/liyanguo/MyImmuCell/02_Read_QC/scRNA_MyImmuCell_high_nCount_RNA.h5ad",compression="gzip")

In [ ]:
print ('scRNA Done.')

In [ ]:
del merged_adata

# 4. Concat and QC raw scADT data for dsb

In [ ]:
files = glob('/home/liyanguo/MyImmuCell/02_Read_QC/MyImmuCell_ADT_h5ad/*_ADT_raw.h5ad')

In [ ]:
files[1:10]

In [ ]:
len(files)

In [ ]:
# Observation

In [ ]:
adata = sc.read_h5ad(files[1])

In [ ]:
sc.pp.calculate_qc_metrics(adata, inplace=True, percent_top=None,log1p=True)

In [ ]:
adata.obs

In [ ]:
sns.displot(
    adata.obs.query("total_counts > 0 and total_counts < 41").total_counts
)

In [ ]:
sns.displot(adata.obs.log1p_total_counts)

In [ ]:
sns.displot(adata.obs.n_genes_by_counts)

In [ ]:
data_dic = {}
for i in tqdm(files):
    sample_name = i.split('/')[-1].split('.')[0]
    adata = sc.read_h5ad(i)
    sc.pp.calculate_qc_metrics(adata, inplace=True, percent_top=None,log1p=True)
    adata.obs["surely_empty_droplet"] = (adata.obs["total_counts"] < 41) & (adata.obs["total_counts"] > 0) & (adata.obs["n_genes_by_counts"] < 5) & (adata.obs["n_genes_by_counts"] > 0)
    adata.obs["surely_empty_droplet"].value_counts()
    adata = adata[adata.obs.surely_empty_droplet].copy()

    # random choice 25% for dsb, about 10 times the high-quality cell number
    np.random.seed(1)
    sample_idx = np.random.choice(adata.n_obs,
                                  size=adata.n_obs//4,
                                  replace=False)
    adata = adata[sample_idx, :].copy()
    data_dic[sample_name] = adata

In [ ]:
adata = sc.concat(data_dic)

In [ ]:
adata

In [ ]:
adata.obs

In [ ]:
adata.write_h5ad("/home/liyanguo/MyImmuCell/02_Read_QC/scADT_raw_MyImmuCell.h5ad",compression="gzip")

In [ ]:
print ('scADT raw Done.')

In [ ]:
del adata,data_dic